# 03 - Labeling

Create long-only binary triple-barrier labels and validate label quality before training.

In [ ]:
import torch, sys
if torch.cuda.is_available():
    print(f"GPU available (labeling is CPU-bound): {torch.cuda.get_device_name(0)}")
else:
    print("CPU mode: notebook 03 does not require a GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = os.environ.get('YENIBOT_REPO_BRANCH', 'research/next-candidate-v1')
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
repo_branch = subprocess.check_output(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True).strip()
assert repo_branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, found {repo_branch}'
sys.path.insert(0, REPO_DIR)
print('Repository branch:', repo_branch)
print('Repository commit:', repo_commit)
print('After a changed checkout, use Runtime -> Restart session before trusting imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
research = cfg.get('experiments', {}).get('next_research_cycle', {})
print('Research cycle:', research.get('status'))
assert research.get('same_window_selection_allowed') is False

In [ ]:
import pandas as pd
from yenibot.labeling import add_long_only_labels, validate_label_quality

features_path = f'{DATA_DIR}/processed/features_1h.parquet'
if not os.path.exists(features_path):
    raise FileNotFoundError(f'Missing feature input; run notebook 02 first: {features_path}')
features = pd.read_parquet(features_path)
assert features['timestamp'].is_monotonic_increasing and not features['timestamp'].duplicated().any()
label_cfg = cfg['labeling']
labeled = add_long_only_labels(
    features,
    atr_column=label_cfg['atr_column'],
    tp_multiplier=label_cfg['tp_multiplier'],
    sl_multiplier=label_cfg['sl_multiplier'],
    max_holding_bars=label_cfg['max_holding_bars'],
)
quality = validate_label_quality(
    labeled,
    min_long_forward_return=label_cfg['min_long_forward_return'],
    max_not_long_forward_return=label_cfg['max_not_long_forward_return'],
    min_long_pct=label_cfg['min_long_pct'],
    max_long_pct=label_cfg['max_long_pct'],
)
print(quality)
print(labeled['label'].value_counts(normalize=True).sort_index())
print(labeled['hit_type'].value_counts(normalize=True))
out = f'{DATA_DIR}/processed/labeled_1h.parquet'
labeled.to_parquet(out, index=False)
print('Saved', out)
print('Labeled date range:', labeled['timestamp'].min(), '->', labeled['timestamp'].max())
print('Labeling complete. Notebook 04 may now run historical training/recency research.')

In [ ]:
import matplotlib.pyplot as plt
labeled.groupby('label')['fwd_return_10h'].mean().plot(kind='bar', title='Forward return by class')
plt.show()
labeled.set_index('timestamp')['label'].rolling(168).mean().plot(title='Rolling long-label rate')
plt.show()

In [ ]:
AUTO_UNASSIGN = False
if AUTO_UNASSIGN:
    from google.colab import runtime
    runtime.unassign()